# LightGBM quantile model (mFRR daily and weekly)

Monthly walk-forward as the baseline and the Ridge. Before each test month, for each `freq × direction`
group, three LightGBM models (p10, p50, p90) are trained on every auction already published. The p50 is the prediction; p10 / p90 give the uncertainty.

- **Validation**: a validation period **before 2025**, to compare with the baseline and the Ridge.
- **Evaluation** (from 2025): produced once by `uv run python -m src.models.lgbm`.

LightGBM trains 3 quantiles × 4 groups × 12 months, with the number of trees chosen on the last 3 months of
each training set.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.data import build_auctions
from src.evaluation import Evaluator
from src.features.build_features import add_model_features, build_features
from src.models import ridge
from src.models.baseline import PreviousKnownAuctionBaseline
from src.models.lgbm import lgbm_quantile_model, to_contract, walk_forward_lgbm
from src.features.build_features import (
    LAG_FEATURES, RATIO_FEATURES, add_lag_features, add_model_features, build_features,  
    OTHER_DIRECTION_FEATURES, WEEKLY_FEATURES, add_other_direction_features, add_weekly_features,
)

BEST = "background-color: #c6efce; color: #006100; font-weight: bold"   # best value (green)

def _fmt(metric):
    if metric.startswith("pinball"):
        return "{:.4f}"
    if metric.startswith("revenue_share") or metric == "share":
        return "{:.1%}"
    if metric.startswith("revenue"):
        return "{:,.0f}"
    return "{:.3f}"


def _best(values, metric):
    v = pd.to_numeric(pd.Series(values), errors="coerce")
    if metric.startswith("coverage"):
        score = (v - 0.80).abs()
    elif metric.startswith(("mae", "rmse", "pinball")):
        score = v
    else:                                  
        score = -v
    return [BEST if s == score.min() else "" for s in score]

def green(table, metric=None, axis=1):
    if metric is not None:
        return table.style.apply(lambda s: _best(s, metric), axis=axis).format(_fmt(metric), na_rep="")
    styled = table.style
    for m in table.columns.get_level_values(0).unique():
        cols = pd.IndexSlice[:, [m]]
        styled = styled.apply(lambda s, m=m: _best(s, m), axis=axis, subset=cols).format(_fmt(m), subset=cols, na_rep="")
    return styled

def green_long(df, metrics, by=("freq", "direction")):
    def styles(d):
        out = pd.DataFrame("", index=d.index, columns=d.columns)
        for m in metrics:
            for idx in d.groupby(list(by)).groups.values():
                out.loc[idx, m] = _best(d.loc[idx, m], m)
        return out
    styled = df.style.apply(styles, axis=None)
    for m in metrics:
        styled = styled.format(_fmt(m), subset=[m], na_rep="")
    return styled

def _gaps(t, metric):
    v = t.apply(pd.to_numeric, errors="coerce")
    if metric.startswith("coverage"):
        d = (v - 0.80).abs()
        return d.sub(d.min(axis=1), axis=0)
    if metric.startswith(("mae", "rmse", "pinball")):
        best = v.min(axis=1)
        return v.sub(best, axis=0).div(best, axis=0)
    best = v.max(axis=1)
    return v.rsub(best, axis=0).div(best, axis=0)

def mean_gap_table(table, metric=None):
    metrics = [metric] if metric is not None else list(table.columns.get_level_values(0).unique())
    out = {}
    for m in metrics:
        t = table if metric is not None else table[m]
        out[m] = _gaps(t, m).mean()
    out = pd.DataFrame(out)
    out.index.name = "mean gap to best"
    fmt = {m: "{:.3f}" if m.startswith("coverage") else "{:.1%}" for m in metrics}
    return out.style.highlight_min(axis=0, props=BEST).format(fmt, na_rep="")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
auctions_all = ridge.prepare_auctions(build_auctions(product="mFRR"), ("daily", "weekly"))
feats = add_model_features(build_features(auctions_all))    
FEATURE_COLS = ridge.model_feature_columns(feats)

print("model inputs:", FEATURE_COLS)
feats.groupby(["freq", "direction"]).size()

model inputs: ['lag1', 'lag2', 'lag3', 'block_00_04', 'block_04_08', 'block_08_12', 'block_12_16', 'block_16_20', 'block_20_24', 'daytype_saturday', 'daytype_sunday_holiday', 'daytype_workday']


freq    direction
daily   downwards    17040
        upwards      17034
weekly  downwards      410
        upwards        410
dtype: int64

## 1. Validation period

The validation period is from 2022-01-01 to 2024-12-31.

In [22]:
a = auctions_all.copy()
a["year"] = a["gc"].dt.year
a.pivot_table(index="year", columns=["freq", "direction"], values="y",
              aggfunc=lambda y: y.isna().mean()).round(2)

freq          daily            weekly        
direction downwards upwards downwards upwards
year                                         
2018           0.00    0.00       NaN     NaN
2019           0.05    0.00       0.0     0.0
2020           0.00    0.06       0.0     0.0
2021           0.11    0.06       0.0     0.0
2022           0.16    0.10       0.0     0.0
2023           0.19    0.12       0.0     0.0
2024           0.90    0.83       0.0     0.0
2025           0.00    0.00       0.0     0.0
2026           0.03    0.00       0.0     0.0

In [23]:
VAL_START, VAL_LAST_DAY = "2022-01-01", "2024-12-31"
VAL_END = (pd.Timestamp(VAL_LAST_DAY) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")   # exclusive end, for the Evaluator

in_val = (auctions_all["gc"] >= pd.Timestamp(VAL_START, tz="UTC")) & (auctions_all["gc"] < pd.Timestamp(VAL_END, tz="UTC"))
print(VAL_START, "to", VAL_LAST_DAY, ":", in_val.sum(), "auctions,", auctions_all.loc[in_val, "y"].notna().sum(), "with a price")

2022-01-01 to 2024-12-31 : 13064 auctions, 8349 with a price


## 2. Walk-forward on the validation period

LightGBM with the default target `delta` = y − lag1.

In [24]:
preds_lgbm = walk_forward_lgbm(feats, start_test=VAL_START, end_test=VAL_LAST_DAY, feature_cols=FEATURE_COLS)
print(len(preds_lgbm), "auctions predicted")

13064 auctions predicted


In [25]:
RIDGE_VARIANT = "ridge_allfeats_delta"      # the best variant of ridge.ipynb
ridge_variants = {
    "ridge_lags3":          dict(feature_cols=ridge.FEATURES, target="level"),
    "ridge_lags3_delta":    dict(feature_cols=ridge.FEATURES, target="delta"),
    "ridge_allfeats_delta": dict(feature_cols=FEATURE_COLS, target="delta"),
    "ridge_lags3_delta_per_series": dict(feature_cols=ridge.FEATURES, target="delta", group_cols=["series"]),
}
preds_ridge = ridge.walk_forward_ridge(feats, start_test=VAL_START, end_test=VAL_LAST_DAY, **ridge_variants[RIDGE_VARIANT])

## 3. LightGBM vs Ridge vs baseline

In [26]:
baseline = PreviousKnownAuctionBaseline().fit(auctions_all)
contracts = {
    "baseline": baseline.predict(auctions_all[in_val]),
    RIDGE_VARIANT: ridge.to_contract(preds_ridge, RIDGE_VARIANT),
    "lgbm_quantile": to_contract(preds_lgbm),
}
evaluator = Evaluator(actuals=auctions_all, start_test=VAL_START, end_test=VAL_END)
report = evaluator.evaluate_many(contracts)

mae_table = report.pivot_table(index=["freq", "direction"], columns="model", values="mae")
display(green(mae_table, "mae"))
display(mean_gap_table(mae_table, "mae"))

,mae
mean gap to best,
baseline,13.3%
lgbm_quantile,0.0%
ridge_allfeats_delta,7.8%


In [ ]:
metrics = [c for c in ["mae", "rmse", "pinball_q10", "pinball_q50", "pinball_q90", "coverage_80"] if c in report.columns]

full = report.pivot_table(index=["freq", "direction"], columns="model", values=metrics)
full = full[metrics]                       # keep the metrics in this order (pivot_table sorts them)
display(green(full))
display(mean_gap_table(full))

# number of auctions scored in each group (the same for every model)
report.groupby(["freq", "direction"])["n"].first().to_frame()

,mae,rmse,pinball_q10,pinball_q50,pinball_q90,coverage_80
mean gap to best,,,,,,
baseline,13.3%,10.3%,,,,
lgbm_quantile,0.0%,2.3%,1.3%,0.0%,3.6%,0.006
ridge_allfeats_delta,7.8%,0.2%,9.4%,7.8%,2.6%,0.025


n
freq   direction      
daily  downwards  3791
       upwards    4244
weekly downwards   157
       upwards     157

In [28]:
rows = {}
for name, p, col in [("baseline", preds_lgbm, "pred_baseline"), (RIDGE_VARIANT, preds_ridge, "pred_ridge"),
                     ("lgbm_quantile", preds_lgbm, "pred_lgbm")]:
    s = p[p["y"].notna()]
    rows[name] = (s[col] - s["y"]).abs().groupby([s["gc"].dt.year, s["freq"], s["direction"]]).mean()
mae_years = pd.DataFrame(rows)
display(green(mae_years, "mae"))
display(mean_gap_table(mae_years, "mae"))

,mae
mean gap to best,
baseline,13.5%
ridge_allfeats_delta,8.6%
lgbm_quantile,0.0%


## 4. Number of trees chosen

For each month, group and quantile, `walk_forward_lgbm` chooses the number of trees on the last 3 months of the
training data.

In [29]:
tuning = preds_lgbm.attrs["tuning"]
tuning.groupby(["freq", "direction", "quantile"])["n_estimators"].value_counts().unstack(fill_value=0)

n_estimators               100  300  600
freq   direction quantile               
daily  downwards 0.1        17    5   14
                 0.5        30    4    2
                 0.9        20    2   14
       upwards   0.1        13   11   12
                 0.5        14   10   12
                 0.9        23   12    1
weekly downwards 0.1        36    0    0
                 0.5        36    0    0
                 0.9        36    0    0
       upwards   0.1        36    0    0
                 0.5        36    0    0
                 0.9        36    0    0

## 5. Feature Importance

One p50 model per group, trained on everything published before the validation period. The importance is related to the number of
times a feature is used in a split.

In [30]:
importance = {}
before = feats[feats["avail"] < VAL_START].dropna(subset=["y", "lag1"])
for (freq, direction), d in before.groupby(["freq", "direction"]):
    model = lgbm_quantile_model(0.5, 300).fit(d[FEATURE_COLS], d["y"] - d["lag1"])
    importance[f"{freq} {direction}"] = pd.Series(model.feature_importances_, index=FEATURE_COLS)

pd.DataFrame(importance).sort_values("daily downwards", ascending=False)

,daily downwards,daily upwards,weekly downwards,weekly upwards
lag1,1245,1187,58,60
lag3,1233,1268,131,150
lag2,1187,1198,111,90
daytype_workday,104,119,0,0
block_00_04,78,106,0,0
block_04_08,69,42,0,0
block_08_12,67,56,0,0
block_12_16,49,64,0,0
block_16_20,45,37,0,0
daytype_sunday_holiday,42,35,0,0


## 6. Quantiles and bidding

In a pay-as-bid auction you are accepted only if your price is at or below the marginal price,
so bidding too high costs everything and bidding too low only part. `p10_lgbm` is a price the actual
marginal price falls below only 10% of the time, `p90_lgbm` one it falls below 90% of the time.

- **coverage** of [p10, p90]: should be close to 80%;
- **share_below_p10** / **share_above_p90**: how often the actual price falls below the p10 / above the
  p90 (both should be close to 10%);
- **mean_width_p10_p90**: mean width of the interval in log10 (narrower is better, at equal coverage);
- **share_rejected**: how often the bid is above the actual price (for p10 it should be close to 10%);
- **revenue_share**: revenue of bidding this value in every auction (paid the bid when accepted, nothing
  when rejected), divided by the revenue of bidding exactly the marginal price.

In [31]:
p = preds_lgbm[preds_lgbm["y"].notna()]
groups = [p["freq"], p["direction"]]

calibration = pd.DataFrame({
    "coverage_p10_p90": ((p["y"] >= p["p10_lgbm"]) & (p["y"] <= p["p90_lgbm"])).groupby(groups).mean(),
    "share_below_p10": (p["y"] < p["p10_lgbm"]).groupby(groups).mean(),
    "share_above_p90": (p["y"] > p["p90_lgbm"]).groupby(groups).mean(),
    "mean_width_p10_p90": (p["p90_lgbm"] - p["p10_lgbm"]).groupby(groups).mean(),
})
calibration.round(3)

coverage_p10_p90  share_below_p10  share_above_p90  \
freq   direction                                                       
daily  downwards             0.762            0.104            0.134   
       upwards               0.770            0.072            0.157   
weekly downwards             0.732            0.153            0.115   
       upwards               0.841            0.108            0.051   

                  mean_width_p10_p90  
freq   direction                      
daily  downwards               0.526  
       upwards                 0.693  
weekly downwards               0.270  
       upwards                 0.175

In [32]:
bids = {"baseline (lag1)": "pred_baseline", "lgbm p50": "p50_lgbm", "lgbm p10": "p10_lgbm"}
perfect = (10 ** p["y"]).groupby(groups).sum()
rows = {}
for name, col in bids.items():
    accepted = p[col] <= p["y"]
    revenue = (10 ** p[col]).where(accepted, 0.0)
    rows[(name, "share_rejected")] = (~accepted).groupby(groups).mean()
    rows[(name, "revenue_share")] = revenue.groupby(groups).sum() / perfect
bidding = pd.DataFrame(rows).T
revenue_rows = pd.IndexSlice[pd.IndexSlice[:, "revenue_share"], :]      # green: best revenue share of each group
bidding.style.apply(lambda s: _best(s, "revenue_share"), axis=0, subset=revenue_rows).format("{:.3f}")

## 7. Adding Features

In [34]:
feats_x = add_lag_features(feats)
FEATURE_COLS_X = FEATURE_COLS + LAG_FEATURES

feats_r = add_lag_features(add_model_features(build_features(auctions_all, ratio_lags=1)))
FEATURE_COLS_R = FEATURE_COLS_X + RATIO_FEATURES

feats_o = add_other_direction_features(feats_x, auctions_all)
feats_w = add_weekly_features(feats_x, auctions_all)
FEATURE_COLS_O = FEATURE_COLS_X + OTHER_DIRECTION_FEATURES
FEATURE_COLS_W = FEATURE_COLS_X + WEEKLY_FEATURES

feats_w[WEEKLY_FEATURES + OTHER_DIRECTION_FEATURES[:0]].describe()

,weekly_lag1,weekly_gap
count,34856.000000,34856.000000
mean,2.754535,1.702061
std,0.552790,0.676933
min,1.455758,-0.405322
25%,2.322219,1.331227
50%,2.703291,1.719451
75%,3.139123,2.106629
max,4.250486,4.765917


In [35]:
preds_lgbm_x = walk_forward_lgbm(feats_x, start_test=VAL_START, end_test=VAL_LAST_DAY,
                                 feature_cols=FEATURE_COLS_X)
preds_lgbm_r = walk_forward_lgbm(feats_r, start_test=VAL_START, end_test=VAL_LAST_DAY,
                                 feature_cols=FEATURE_COLS_R)
preds_lgbm_o = walk_forward_lgbm(feats_o, start_test=VAL_START, end_test=VAL_LAST_DAY, feature_cols=FEATURE_COLS_O)
preds_lgbm_w = walk_forward_lgbm(feats_w, start_test=VAL_START, end_test=VAL_LAST_DAY, feature_cols=FEATURE_COLS_W)

In [ ]:
contracts_x = {
    "baseline": contracts["baseline"],
    "lgbm_quantile": contracts["lgbm_quantile"],            
    "lgbm_lagfeats": to_contract(preds_lgbm_x, "lgbm_lagfeats"),
}
report_x = evaluator.evaluate_many(contracts_x)
table = report_x.pivot_table(index=["freq", "direction"], columns="model",
                        values=["mae", "pinball_q10", "coverage_80"])
display(green(table))
display(mean_gap_table(table))

,coverage_80,mae,pinball_q10
mean gap to best,,,
baseline,,14.6%,
lgbm_lagfeats,0.009,0.0%,0.0%
lgbm_quantile,0.001,1.1%,8.1%


In [37]:
contracts_r = {
    "baseline": contracts["baseline"],
    "lgbm_quantile": contracts["lgbm_quantile"],
    "lgbm_lagfeats": to_contract(preds_lgbm_x, "lgbm_lagfeats"),
    "lgbm_ratio": to_contract(preds_lgbm_r, "lgbm_ratio"),
}
report_r = evaluator.evaluate_many(contracts_r)
table = report_r.pivot_table(index=["freq", "direction"], columns="model",
                        values=["mae", "pinball_q10", "coverage_80"])
display(green(table))
display(mean_gap_table(table))

,coverage_80,mae,pinball_q10
mean gap to best,,,
baseline,,14.8%,
lgbm_lagfeats,0.012,0.2%,0.2%
lgbm_quantile,0.004,1.3%,8.3%
lgbm_ratio,0.014,0.5%,0.3%


In [38]:
contracts_c = {
    "baseline": contracts["baseline"],
    "lgbm_lagfeats": to_contract(preds_lgbm_x, "lgbm_lagfeats"),
    "lgbm_otherdir": to_contract(preds_lgbm_o, "lgbm_otherdir"),
    "lgbm_weekly": to_contract(preds_lgbm_w, "lgbm_weekly"),
}
report_c = evaluator.evaluate_many(contracts_c)
table = report_c.pivot_table(index=["freq", "direction"], columns="model",
                        values=["mae", "pinball_q10", "coverage_80"])
display(green(table))
display(mean_gap_table(table))

,coverage_80,mae,pinball_q10
mean gap to best,,,
baseline,,16.2%,
lgbm_lagfeats,0.007,1.4%,2.6%
lgbm_otherdir,0.007,0.5%,0.5%
lgbm_weekly,0.013,0.7%,0.8%


## Conclusion

- **LightGBM beats both references in almost every group.** MAE of the first LightGBM (`lgbm_quantile`)
  vs baseline vs ridge: 0.201 vs 0.228 vs 0.209 (daily downwards), 0.239 vs 0.292 vs 0.267 (daily upwards),
  0.099 vs 0.106 vs 0.107 (weekly downwards), 0.036 vs 0.040 vs 0.039 (weekly upwards). 
- **Quantiles.** The p10 is better than the ridge one in the daily groups (pinball 0.042 vs 0.049
  downwards, 0.051 vs 0.060 upwards) and similar in the weekly ones. The intervals are narrower than the
  ridge ones and a bit too narrow: coverage 0.73–0.84 (target 0.80), against 0.76–0.89 for the ridge.
- **Number of trees.** In the daily groups the inner validation picks 100, 300 or 600 trees depending on
  the month; in the weekly groups always 100.
- **What the model uses.** Mostly the three lags; then the day type in the daily groups (mainly
  workday vs weekend); the blocks much less, since they are already in the lags of the same series.
- **Bidding.** Bidding the p10 earns the most in every group: 42–60% of the perfect revenue in
  daily downwards / weekly downwards, 38% and 79% in daily upwards / weekly upwards, against 18–29% for
  the baseline. The p10 is rejected 7–15% of the time, close to the expected 10%.
- **Features.** Two additions are kept: the lag features (`mom1`, `mom2`, `lag_mean`, `lag_std`), better
  in every group
  (`other_lag1`, `other_gap`), better or equal in every group. The
  offered / awarded ratio and the weekly price for daily auctions give no consistent gain.
- **Final model: `lgbm_otherdir`** (lags, block and day-type one-hots, lag features, other-direction
  features). 